# ACF and PACF Theory

The **autocorrelation function (ACF)** and **partial autocorrelation function
(PACF)** are the two most important diagnostic tools in time series analysis.
They reveal the internal correlation structure of a series and are the primary
means of identifying appropriate ARIMA model orders.

This notebook covers:
1. The ACF formula and manual computation
2. Partial autocorrelation and the PACF
3. Yule-Walker equations (overview)
4. `plot_acf()` and `plot_pacf()` from statsmodels
5. The ACF/PACF signature table for model identification
6. Application to real datasets

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Airline Passengers (monthly, 1949-1960) ---
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# --- Daily Total Female Births (daily, 1959) ---
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]

print("Airline shape:", airline.shape)
print("Births shape: ", births.shape)
airline.head()

---
## 1. Autocorrelation Function (ACF)

### Definition

The **autocorrelation at lag $k$** measures the linear association between
observations that are $k$ time steps apart:

$$
r_k = \frac{\sum_{t=k+1}^{T} (y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{T} (y_t - \bar{y})^2}
$$

Key properties:
- $r_0 = 1$ always (a series is perfectly correlated with itself)
- $-1 \le r_k \le 1$
- $r_k = r_{-k}$ (symmetric)
- The denominator uses *all* $T$ observations, not just $T - k$, which is a
  deliberate choice that ensures the ACF matrix is positive semi-definite

### Manual ACF Computation

Let's compute the ACF by hand for a small example to build intuition, then
verify against `statsmodels`.

In [ ]:
# Small example: first 12 months of airline passengers
y = airline["Passengers"].values[:12].astype(float)
T = len(y)
y_bar = y.mean()

print(f"Series (first 12 months): {y}")
print(f"Mean: {y_bar:.2f}")
print(f"T = {T}")

In [ ]:
# Step 1: compute the denominator — total sum of squared deviations
denominator = np.sum((y - y_bar) ** 2)
print(f"Denominator (total SS): {denominator:.2f}")
print()

# Step 2: compute numerator for each lag k
max_lag = 6
manual_acf = np.zeros(max_lag + 1)

for k in range(max_lag + 1):
    numerator = np.sum((y[k:] - y_bar) * (y[:T - k] - y_bar)) if k > 0 else denominator
    manual_acf[k] = numerator / denominator
    print(f"  r_{k} = {numerator:10.2f} / {denominator:.2f} = {manual_acf[k]:.4f}")

In [ ]:
# Verify against statsmodels
sm_acf = acf(y, nlags=max_lag, fft=False)

comparison = pd.DataFrame({
    "Lag": range(max_lag + 1),
    "Manual ACF": manual_acf,
    "statsmodels ACF": sm_acf,
    "Difference": np.abs(manual_acf - sm_acf),
})
print(comparison.to_string(index=False))
print("\nPerfect match — our manual formula agrees with statsmodels.")

### Interpreting the ACF

- **Slowly decaying ACF** suggests a trend (the series drifts, so nearby
  values stay correlated for many lags).
- **Periodic spikes** at seasonal lags (e.g., lag 12 for monthly data)
  indicate seasonality.
- **Sharp cutoff** after lag $q$ suggests an MA($q$) process.
- Values within the **confidence band** (typically $\pm 1.96 / \sqrt{T}$)
  are not statistically significant.

In [ ]:
# Confidence band width
T_full = len(airline)
ci = 1.96 / np.sqrt(T_full)
print(f"Approximate 95% confidence band: +/- {ci:.4f}")
print(f"(Based on T = {T_full} observations)")

---
## 2. Partial Autocorrelation Function (PACF)

### The problem with ACF

The ACF at lag $k$ captures **both** the direct correlation between $y_t$ and
$y_{t-k}$ **and** the indirect correlation transmitted through intermediate
lags $y_{t-1}, y_{t-2}, \ldots, y_{t-k+1}$.

For model identification we often need to know the **direct** (partial)
correlation only.

### Definition

The **partial autocorrelation** at lag $k$ is the correlation between $y_t$
and $y_{t-k}$ **after removing** (regressing out) the linear dependence on the
intervening observations $y_{t-1}, y_{t-2}, \ldots, y_{t-k+1}$.

Formally, if we regress $y_t$ on $y_{t-1}, \ldots, y_{t-k}$:

$$
y_t = \phi_{k,1} y_{t-1} + \phi_{k,2} y_{t-2} + \cdots + \phi_{k,k} y_{t-k} + \varepsilon_t
$$

then the PACF at lag $k$ is the coefficient $\phi_{k,k}$ — the coefficient on
the *last* lag in the regression.

In [ ]:
# Demonstrate PACF via successive regressions on the births dataset
y_births = births["Births"].values.astype(float)
y_births_centered = y_births - y_births.mean()
n = len(y_births)

# PACF at lag 1 = ACF at lag 1 (nothing to partial out)
pacf_lag1 = np.corrcoef(y_births_centered[1:], y_births_centered[:-1])[0, 1]
print(f"PACF(1) = ACF(1) = {pacf_lag1:.4f}")

# PACF at lag 2: regress y_t on y_{t-1} and y_{t-2}, take coefficient of y_{t-2}
from numpy.linalg import lstsq

X2 = np.column_stack([y_births[1:-1], y_births[:-2]])
y2 = y_births[2:]
coeffs2, _, _, _ = lstsq(X2 - X2.mean(axis=0), y2 - y2.mean(), rcond=None)
print(f"PACF(2) via regression: {coeffs2[1]:.4f}")

# PACF at lag 3: regress y_t on y_{t-1}, y_{t-2}, y_{t-3}
X3 = np.column_stack([y_births[2:-1], y_births[1:-2], y_births[:-3]])
y3 = y_births[3:]
coeffs3, _, _, _ = lstsq(X3 - X3.mean(axis=0), y3 - y3.mean(), rcond=None)
print(f"PACF(3) via regression: {coeffs3[2]:.4f}")

In [ ]:
# Compare with statsmodels PACF
sm_pacf = pacf(y_births, nlags=5, method="ols")

print("statsmodels PACF (OLS method):")
for k, val in enumerate(sm_pacf):
    print(f"  PACF({k}) = {val:.4f}")

### Yule-Walker Equations

An alternative (and more efficient) way to compute the PACF is through the
**Yule-Walker equations**, which express the autocorrelations of an AR($p$)
process as a system of linear equations:

$$
\begin{bmatrix} r_1 \\ r_2 \\ \vdots \\ r_k \end{bmatrix}
=
\begin{bmatrix}
1 & r_1 & r_2 & \cdots & r_{k-1} \\
r_1 & 1 & r_1 & \cdots & r_{k-2} \\
\vdots & & \ddots & & \vdots \\
r_{k-1} & r_{k-2} & \cdots & r_1 & 1
\end{bmatrix}
\begin{bmatrix} \phi_{k,1} \\ \phi_{k,2} \\ \vdots \\ \phi_{k,k} \end{bmatrix}
$$

Solving for $\phi_{k,k}$ at each $k$ gives the PACF.  The Toeplitz structure
of the matrix can be exploited for efficient computation (Levinson-Durbin
recursion).  We will not derive the full solution here — the key takeaway is
that `statsmodels` uses this approach by default (`method='ywadjusted'` or
`method='ywmle'`).

In [ ]:
# Yule-Walker PACF via statsmodels
sm_pacf_yw = pacf(y_births, nlags=5, method="ywadjusted")

print("Yule-Walker PACF:")
for k, val in enumerate(sm_pacf_yw):
    print(f"  PACF({k}) = {val:.4f}")

print("\nNote: slight differences from OLS method are normal — both are valid estimators.")

---
## 3. Plotting ACF and PACF with statsmodels

`statsmodels` provides `plot_acf()` and `plot_pacf()` which produce
publication-quality correlogram plots with confidence bands.

In [ ]:
# ACF and PACF of the Births dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(births["Births"], lags=40, ax=axes[0], title="ACF — Daily Female Births")
plot_pacf(births["Births"], lags=40, ax=axes[1], title="PACF — Daily Female Births",
          method="ywm")

plt.tight_layout()
plt.show()

print("The PACF cuts off sharply after a few lags, while the ACF decays slowly.")
print("This pattern suggests an AR process (see signature table below).")

In [ ]:
# ACF and PACF of Airline Passengers — a trending, seasonal series
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(airline["Passengers"], lags=48, ax=axes[0],
         title="ACF — Airline Passengers")
plot_pacf(airline["Passengers"], lags=48, ax=axes[1],
          title="PACF — Airline Passengers", method="ywm")

plt.tight_layout()
plt.show()

print("The ACF decays very slowly — a hallmark of non-stationarity (trend).")
print("Periodic humps at lags 12, 24, 36 reflect the 12-month seasonality.")

### Understanding the confidence bands

The shaded blue region in `plot_acf` / `plot_pacf` represents the approximate
95% confidence interval under the null hypothesis that the true autocorrelation
is zero.  The default formula is:

$$
\pm \frac{z_{\alpha/2}}{\sqrt{T}} = \pm \frac{1.96}{\sqrt{T}}
$$

Any spike that extends beyond the band is statistically significant at the 5%
level.  With $T = 144$ (airline data), the band width is $\pm 0.163$.

In [ ]:
# Manually annotate the confidence band
T_airline = len(airline)
T_births = len(births)

print(f"Airline passengers: T = {T_airline}, CI = +/- {1.96 / np.sqrt(T_airline):.4f}")
print(f"Daily births:       T = {T_births}, CI = +/- {1.96 / np.sqrt(T_births):.4f}")
print()
print("More data => narrower band => easier to detect small correlations.")

---
## 4. Simulated Examples — Pure AR, MA, and ARMA

To build intuition for the signature table, let's simulate known processes
and examine their ACF/PACF.

In [ ]:
# Simulate AR(1): y_t = 0.8 * y_{t-1} + eps
np.random.seed(42)
n_sim = 500
eps = np.random.randn(n_sim)

ar1 = np.zeros(n_sim)
for t in range(1, n_sim):
    ar1[t] = 0.8 * ar1[t - 1] + eps[t]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(ar1)
axes[0].set_title("AR(1) with $\\phi_1 = 0.8$")
axes[0].set_xlabel("Time")

plot_acf(ar1, lags=30, ax=axes[1], title="ACF — AR(1)")
plot_pacf(ar1, lags=30, ax=axes[2], title="PACF — AR(1)", method="ywm")

plt.tight_layout()
plt.show()

print("AR(1) signature: ACF decays exponentially, PACF cuts off after lag 1.")

In [ ]:
# Simulate AR(2): y_t = 0.5 * y_{t-1} + 0.3 * y_{t-2} + eps
np.random.seed(42)

ar2 = np.zeros(n_sim)
for t in range(2, n_sim):
    ar2[t] = 0.5 * ar2[t - 1] + 0.3 * ar2[t - 2] + eps[t]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(ar2)
axes[0].set_title("AR(2) with $\\phi_1 = 0.5$, $\\phi_2 = 0.3$")
axes[0].set_xlabel("Time")

plot_acf(ar2, lags=30, ax=axes[1], title="ACF — AR(2)")
plot_pacf(ar2, lags=30, ax=axes[2], title="PACF — AR(2)", method="ywm")

plt.tight_layout()
plt.show()

print("AR(2) signature: ACF decays (mix of exponentials), PACF cuts off after lag 2.")

In [ ]:
# Simulate MA(1): y_t = eps_t + 0.7 * eps_{t-1}
np.random.seed(42)

ma1 = np.zeros(n_sim)
for t in range(1, n_sim):
    ma1[t] = eps[t] + 0.7 * eps[t - 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(ma1)
axes[0].set_title("MA(1) with $\\theta_1 = 0.7$")
axes[0].set_xlabel("Time")

plot_acf(ma1, lags=30, ax=axes[1], title="ACF — MA(1)")
plot_pacf(ma1, lags=30, ax=axes[2], title="PACF — MA(1)", method="ywm")

plt.tight_layout()
plt.show()

print("MA(1) signature: ACF cuts off after lag 1, PACF decays (tails off).")

In [ ]:
# Simulate MA(2): y_t = eps_t + 0.7 * eps_{t-1} + 0.4 * eps_{t-2}
np.random.seed(42)
eps2 = np.random.randn(n_sim)

ma2 = np.zeros(n_sim)
for t in range(2, n_sim):
    ma2[t] = eps2[t] + 0.7 * eps2[t - 1] + 0.4 * eps2[t - 2]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(ma2)
axes[0].set_title("MA(2) with $\\theta_1 = 0.7$, $\\theta_2 = 0.4$")
axes[0].set_xlabel("Time")

plot_acf(ma2, lags=30, ax=axes[1], title="ACF — MA(2)")
plot_pacf(ma2, lags=30, ax=axes[2], title="PACF — MA(2)", method="ywm")

plt.tight_layout()
plt.show()

print("MA(2) signature: ACF cuts off after lag 2, PACF tails off.")

In [ ]:
# Simulate ARMA(1,1): y_t = 0.7 * y_{t-1} + eps_t + 0.4 * eps_{t-1}
np.random.seed(42)
eps3 = np.random.randn(n_sim)

arma11 = np.zeros(n_sim)
for t in range(1, n_sim):
    arma11[t] = 0.7 * arma11[t - 1] + eps3[t] + 0.4 * eps3[t - 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(arma11)
axes[0].set_title("ARMA(1,1) with $\\phi_1 = 0.7$, $\\theta_1 = 0.4$")
axes[0].set_xlabel("Time")

plot_acf(arma11, lags=30, ax=axes[1], title="ACF — ARMA(1,1)")
plot_pacf(arma11, lags=30, ax=axes[2], title="PACF — ARMA(1,1)", method="ywm")

plt.tight_layout()
plt.show()

print("ARMA(1,1) signature: BOTH ACF and PACF tail off — neither cuts off sharply.")
print("This is the hardest case to identify from ACF/PACF alone.")

---
## 5. ACF/PACF Signature Table for Model Identification

This is the **most important table** in classical time series analysis.  It
summarises how the ACF and PACF behave for the three main process types:

| Model | ACF Pattern | PACF Pattern |
|-------|-------------|---------------|
| AR($p$) | Tails off (decays exponentially or in damped oscillations) | **Cuts off** after lag $p$ |
| MA($q$) | **Cuts off** after lag $q$ | Tails off (decays exponentially or in damped oscillations) |
| ARMA($p$, $q$) | Tails off | Tails off |

### How to use this table

1. Plot the ACF and PACF of the (stationary!) series.
2. **If the PACF cuts off at lag $p$** and the ACF decays: suggest AR($p$).
3. **If the ACF cuts off at lag $q$** and the PACF decays: suggest MA($q$).
4. **If both tail off**: suggest ARMA($p$, $q$) — further analysis (AIC/BIC)
   is needed to determine the orders.

> **Important:** the series must be **stationary** before applying this table.
> A trending or seasonal series will show a slowly-decaying ACF that masks the
> true correlation structure.  See Notebook 02 for stationarity and differencing.

In [ ]:
# Summary of simulated examples
summary = pd.DataFrame({
    "Process": ["AR(1)", "AR(2)", "MA(1)", "MA(2)", "ARMA(1,1)"],
    "ACF": ["Decays exponentially", "Decays (damped)", "Cuts off lag 1",
            "Cuts off lag 2", "Tails off"],
    "PACF": ["Cuts off lag 1", "Cuts off lag 2", "Decays exponentially",
             "Decays (damped)", "Tails off"],
})
print(summary.to_string(index=False))

---
## 6. Application to Real Data

### Daily Female Births — a stationary series

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time plot
axes[0, 0].plot(births["Births"])
axes[0, 0].set_title("Daily Total Female Births (1959)")
axes[0, 0].set_ylabel("Births")

# Distribution
axes[0, 1].hist(births["Births"], bins=25, edgecolor="white", alpha=0.7)
axes[0, 1].set_title("Distribution")
axes[0, 1].set_xlabel("Births")

# ACF
plot_acf(births["Births"], lags=40, ax=axes[1, 0], title="ACF")

# PACF
plot_pacf(births["Births"], lags=40, ax=axes[1, 1], title="PACF", method="ywm")

plt.suptitle("Daily Female Births — ACF/PACF Analysis", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - The series looks stationary (no trend, roughly constant variance).")
print("  - ACF decays gradually — consistent with an AR process.")
print("  - PACF has significant spikes at lags 1-3 and possibly a few more.")
print("  - Tentative model suggestion: AR(2) or AR(3).")

### Airline Passengers — a non-stationary series

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(airline["Passengers"], lags=48, ax=axes[0],
         title="ACF — Airline Passengers (Original)")
plot_pacf(airline["Passengers"], lags=48, ax=axes[1],
          title="PACF — Airline Passengers (Original)", method="ywm")

plt.tight_layout()
plt.show()

print("The slow ACF decay signals non-stationarity.")
print("We should NOT try to identify AR/MA orders from this — difference first!")

In [ ]:
# First + seasonal differencing
airline_diff = airline["Passengers"].diff().diff(12).dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(airline_diff)
axes[0].set_title("Airline: $\\nabla \\nabla_{12} y_t$")
axes[0].set_ylabel("Differenced Passengers")

plot_acf(airline_diff, lags=36, ax=axes[1],
         title="ACF — Differenced Airline")
plot_pacf(airline_diff, lags=36, ax=axes[2],
          title="PACF — Differenced Airline", method="ywm")

plt.tight_layout()
plt.show()

print("After differencing, the ACF/PACF can now be used for model identification.")
print("Significant spike at lag 1 in ACF suggests an MA(1) component.")
print("Spike at lag 12 suggests a seasonal MA(1) component.")
print("This points toward a SARIMA(0,1,1)(0,1,1)_12 — the classic airline model.")

---
## 7. Additional Datasets

In [ ]:
# Energy Production — another seasonal series
energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    index_col="DATE",
    parse_dates=True,
)
energy.index.freq = "MS"
energy.columns = ["Production"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(energy["Production"].dropna(), lags=48, ax=axes[0],
         title="ACF — Energy Production")
plot_pacf(energy["Production"].dropna(), lags=48, ax=axes[1],
          title="PACF — Energy Production", method="ywm")

plt.tight_layout()
plt.show()

print("Similar pattern to airline passengers: slow decay + seasonal humps.")
print("This series also needs differencing before model identification.")

In [ ]:
# White noise — should show NO significant autocorrelation
np.random.seed(123)
white_noise = np.random.randn(500)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(white_noise)
axes[0].set_title("White Noise")

plot_acf(white_noise, lags=30, ax=axes[1], title="ACF — White Noise")
plot_pacf(white_noise, lags=30, ax=axes[2], title="PACF — White Noise", method="ywm")

plt.tight_layout()
plt.show()

print("All spikes lie within the confidence bands — no significant autocorrelation.")
print("This is what 'good' model residuals should look like.")

---
## 8. Numerical ACF/PACF Values

In [ ]:
# Print numerical values for births dataset
acf_vals = acf(births["Births"], nlags=15)
pacf_vals = pacf(births["Births"], nlags=15, method="ywm")

ci_births = 1.96 / np.sqrt(len(births))

acf_df = pd.DataFrame({
    "Lag": range(16),
    "ACF": acf_vals,
    "PACF": pacf_vals,
    "Significant ACF": ["*" if abs(v) > ci_births and k > 0 else "" for k, v in enumerate(acf_vals)],
    "Significant PACF": ["*" if abs(v) > ci_births and k > 0 else "" for k, v in enumerate(pacf_vals)],
})

print(f"95% CI threshold: +/- {ci_births:.4f}")
print()
print(acf_df.to_string(index=False, float_format="{:.4f}".format))

---
## 9. Common Pitfalls

1. **Applying the signature table to non-stationary data.** The slowly
   decaying ACF of a trending series does *not* mean AR(large $p$). It means
   the data must be differenced first.

2. **Ignoring the confidence bands.** A few spikes will exceed the band by
   chance alone (5% of them at the 95% level). Look for *patterns*, not
   isolated spikes.

3. **Using too many lags.** As a rule of thumb, examine at most
   $\min(T/4, 10 \cdot \log_{10}(T))$ lags. Estimates at very high lags are
   unreliable due to few overlapping observations.

4. **Confusing ACF and PACF roles.** Remember: PACF identifies AR order;
   ACF identifies MA order.

In [ ]:
# Recommended max lags
for name, T_val in [("Airline", T_airline), ("Births", T_births)]:
    max_lag_rule = min(T_val // 4, int(10 * np.log10(T_val)))
    print(f"{name}: T = {T_val}, recommended max lags = {max_lag_rule}")

---
## Summary

| Concept | Key Idea |
|---------|----------|
| ACF $r_k$ | Total correlation between $y_t$ and $y_{t-k}$ (direct + indirect) |
| PACF $\phi_{k,k}$ | Direct correlation between $y_t$ and $y_{t-k}$, removing intermediate lags |
| AR($p$) signature | ACF tails off, PACF cuts off at lag $p$ |
| MA($q$) signature | ACF cuts off at lag $q$, PACF tails off |
| ARMA($p$,$q$) signature | Both tail off — use AIC/BIC to choose orders |
| Confidence band | $\pm 1.96 / \sqrt{T}$ — spikes outside are significant |
| Prerequisite | Series **must be stationary** before applying the signature table |

In [ ]:
print("Next: Notebook 02 — Stationarity and Differencing")
print("We will learn how to test for and achieve stationarity,")
print("which is a prerequisite for the model identification approach shown here.")